In [ ]:
# ============================================================
# Colon ablation: dense latent factor vs block-subunit decoders
# Physical GPU ID 3 (masked to cuda:0 inside this process)
#
# Methods (all matched at M=8, flow prior, marginals, missingness,
# synthetic budget, downstream LR, and 5x5 fold-wise evaluation):
#   1) Dense latent factor: Z = H W^T + b(Y) + eps
#      - H inferred by PCA in Gaussian-copula space
#      - each feature loads on all M factors
#   2) BSTabDiff block-sparse decoder without GO-BS
#   3) BSTabDiff block-sparse decoder with GO-BS
#
# Outputs:
#   - fold-level and summary CSVs
#   - full-data fidelity/resource CSVs
#   - paired-comparison CSV
#   - PDF/PNG comparison plots
# ============================================================

import os
os.environ["CUDA_DEVICE_ORDER"] = "PCI_BUS_ID"
os.environ["CUDA_VISIBLE_DEVICES"] = "3"
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
os.environ["PYTHONWARNINGS"] = "ignore"

import gc
import math
import time
import random
import warnings
import importlib.util
import sys
import types
from dataclasses import dataclass
from pathlib import Path
from typing import Dict, Optional, Tuple

import numpy as np
import pandas as pd

warnings.simplefilter("ignore")
from sklearn.exceptions import ConvergenceWarning
warnings.filterwarnings("ignore", category=ConvergenceWarning)
warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", category=RuntimeWarning)

from sklearn.decomposition import PCA
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, roc_auc_score
from sklearn.model_selection import RepeatedStratifiedKFold, StratifiedShuffleSplit
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.feature_selection import mutual_info_classif
from sklearn.neighbors import NearestNeighbors

try:
    from scipy.stats import wilcoxon
except Exception:
    wilcoxon = None

try:
    import psutil
except Exception:
    psutil = None

import torch

# matplotlib is used only for output plots.
import matplotlib.pyplot as plt


# ============================================================
# Configuration
# ============================================================
GLOBAL_SEED = 42
DATA_FILE = "coloncancer_encoded.csv"
TARGET_COL = "label"
MODULE_CANDIDATES = [
    "bstabdiff_gobs.py",
    "bstabdiff_gobs (2).py",
    "/mnt/data/bstabdiff_gobs (2).py",
]

DEVICE = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
M = 8
N_SYN = 200
N_SPLITS = 5
N_REPEATS = 5

# Matched tuned prior settings from the Colon study.
PRIOR_TYPE = "flow"
PRIOR_EPOCHS = 300
PRIOR_BATCH = 64
PRIOR_LR = 0.0032931446587915556
USE_EMA = True
EMA_DECAY = 0.9925524818123974

# Best GO-BS settings used in the fixed Colon evaluation.
GOBS_PARAMS = {
    "gobs_num_clusters": 3,
    "gobs_metric": "correlation",
    "gobs_bins": 24,
    "gobs_top_k": 16,
    "gobs_refine_order": False,
    "gobs_direction_select": False,
    "gobs_refine_passes": 3,
    "gobs_boundary_refine_passes": 5,
    "gobs_boundary_window": 16,
    "gobs_lambda_cross": 2.3988509383361634,
    "gobs_gamma_balance": 5.193972052958215e-06,
}

METHODS = [
    "DenseFactor-PCA",
    "BlockSparse-NoGO-BS",
    "BSTabDiff-GO-BS",
]

OUT_PREFIX = "colon_dense_vs_block_cuda3"


# ============================================================
# Utilities
# ============================================================
def log(msg: str) -> None:
    print(msg, flush=True)


def set_seed(seed: int = GLOBAL_SEED) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def cleanup() -> None:
    gc.collect()
    if torch.cuda.is_available():
        try:
            torch.cuda.empty_cache()
            torch.cuda.ipc_collect()
        except Exception:
            pass


def _install_kmeans_gpu_fallback_if_needed():
    """Provide a CPU-compatible shim only when kmeans_gpu is unavailable."""
    try:
        import kmeans_gpu  # noqa: F401
        return
    except Exception:
        pass

    from sklearn.cluster import KMeans as _SKKMeans

    class _KMeansGPUFallback:
        def __init__(self, n_clusters, max_iter=100, tolerance=1e-4, **kwargs):
            self.n_clusters = int(n_clusters)
            self.max_iter = int(max_iter)
            self.tolerance = float(tolerance)

        def __call__(self, points, features):
            del features
            x = points[0].detach().cpu().numpy()
            km = _SKKMeans(
                n_clusters=min(self.n_clusters, x.shape[0]),
                max_iter=self.max_iter,
                tol=self.tolerance,
                random_state=GLOBAL_SEED,
                n_init=10,
            ).fit(x)
            centroids = torch.from_numpy(km.cluster_centers_.astype(np.float32)).to(points.device)
            labels = torch.from_numpy(km.labels_.astype(np.int64)).to(points.device)
            return centroids.unsqueeze(0), labels.unsqueeze(0)

    shim = types.ModuleType("kmeans_gpu")
    shim.KMeans = _KMeansGPUFallback
    sys.modules["kmeans_gpu"] = shim
    log("kmeans_gpu not found; using sklearn KMeans fallback for clustering.")


def load_bstabdiff_module():
    _install_kmeans_gpu_fallback_if_needed()
    for candidate in MODULE_CANDIDATES:
        p = Path(candidate)
        if p.exists():
            spec = importlib.util.spec_from_file_location("bstabdiff_gobs_local", str(p.resolve()))
            if spec is None or spec.loader is None:
                continue
            module = importlib.util.module_from_spec(spec)
            sys.modules[spec.name] = module
            spec.loader.exec_module(module)
            log(f"Loaded BSTabDiff module from: {p.resolve()}")
            return module
    raise FileNotFoundError(
        "Could not locate bstabdiff_gobs.py. Put it beside this script/notebook "
        "or update MODULE_CANDIDATES."
    )


bstab = load_bstabdiff_module()
FeatureSpec = bstab.FeatureSpec
EmpiricalMarginals = bstab.EmpiricalMarginals
FlowPrior = bstab.FlowPrior
DiffusionPrior = bstab.DiffusionPrior
EMA = bstab.EMA
fit_block_subunit_generator = bstab.fit_block_subunit_generator
erf_normal_cdf = bstab.erf_normal_cdf
approx_normal_icdf = bstab.approx_normal_icdf


def make_lr(seed: int) -> Pipeline:
    return Pipeline(
        [
            ("imp", SimpleImputer(strategy="median")),
            ("sc", StandardScaler(with_mean=True, with_std=True)),
            (
                "clf",
                LogisticRegression(
                    max_iter=10000,
                    class_weight="balanced",
                    solver="lbfgs",
                    random_state=seed,
                ),
            ),
        ]
    )


def positive_class_score(clf, X: np.ndarray) -> np.ndarray:
    if hasattr(clf, "predict_proba"):
        p = clf.predict_proba(X)
        return p[:, 1] if p.ndim == 2 and p.shape[1] == 2 else p.max(axis=1)
    if hasattr(clf, "decision_function"):
        s = clf.decision_function(X)
        return s if np.ndim(s) == 1 else s[:, 1]
    return clf.predict(X).astype(float)


def safe_auc(y_true: np.ndarray, scores: np.ndarray) -> float:
    try:
        return float(roc_auc_score(y_true, scores))
    except Exception:
        return float("nan")


def load_colon_dataset(csv_path: str = DATA_FILE, label_col: str = TARGET_COL):
    df = pd.read_csv(csv_path)
    if label_col not in df.columns:
        raise ValueError(f"Label column '{label_col}' not found.")
    y_raw = df[label_col].to_numpy()
    X_df = df.drop(columns=[label_col]).select_dtypes(include=[np.number])
    X = X_df.to_numpy(dtype=np.float32)
    X[~np.isfinite(X)] = np.nan

    if y_raw.dtype.kind in {"U", "S", "O"}:
        y, _ = pd.factorize(y_raw)
    else:
        uniq = np.unique(y_raw)
        if not np.array_equal(uniq, np.arange(uniq.size)):
            y, _ = pd.factorize(y_raw)
        else:
            y = y_raw.astype(int, copy=False)
    return X, np.asarray(y, dtype=int), list(X_df.columns)


# ============================================================
# Dense latent-factor baseline
# ============================================================
def gaussianize_columns(X: np.ndarray, R: np.ndarray) -> np.ndarray:
    """Rank-Gaussianize each feature using training observations only."""
    X = np.asarray(X, dtype=np.float32)
    R = np.asarray(R, dtype=np.int64)
    n, m = X.shape
    Z = np.zeros((n, m), dtype=np.float32)

    for j in range(m):
        obs = (R[:, j] == 1) & np.isfinite(X[:, j])
        vals = X[obs, j].astype(np.float64)
        if vals.size < 2:
            continue
        order = np.argsort(vals, kind="mergesort")
        ranks = np.empty_like(order)
        ranks[order] = np.arange(vals.size)
        u = (ranks + 0.5) / vals.size
        z = approx_normal_icdf(torch.from_numpy(u.astype(np.float32))).cpu().numpy()
        Z[obs, j] = z.astype(np.float32)
    return Z


@dataclass
class DenseEmissionParams:
    W: torch.Tensor                 # (m, M)
    sigma: torch.Tensor             # (m,)
    b: Optional[torch.Tensor]       # (C, m)
    miss_rate: torch.Tensor         # (m,)
    miss_rate_y: Optional[torch.Tensor]  # (C, m)


class DenseLatentFactorGenerator:
    """
    Matched dense baseline:
        Z_j = W_j h + b_j(Y) + xi_j

    Differences from BSTabDiff are intentionally restricted to:
      - latent inference: global PCA scores in Gaussian-copula space;
      - emission loading: every feature loads on all M factors.

    The latent prior, class-conditional marginals, missingness handling,
    synthetic sampling protocol, and M are matched to BSTabDiff.
    """

    def __init__(
        self,
        feature_specs,
        M: int,
        n_classes: Optional[int],
        prior_type: str,
        device: torch.device,
        use_class_cond_marginals: bool = True,
        use_class_cond_missingness: bool = True,
    ):
        self.feature_specs = feature_specs
        self.m = len(feature_specs)
        self.M = int(M)
        self.n_classes = n_classes
        self.device = torch.device(device)
        self.use_class_cond_marginals = bool(use_class_cond_marginals and n_classes is not None)
        self.use_class_cond_missingness = bool(use_class_cond_missingness and n_classes is not None)

        self.marginals = EmpiricalMarginals(
            m=self.m,
            n_classes=self.n_classes if self.use_class_cond_marginals else None,
            device=self.device,
        )

        if prior_type == "flow":
            self.prior = FlowPrior(M=self.M, n_classes=self.n_classes).to(self.device)
        elif prior_type == "diffusion":
            self.prior = DiffusionPrior(M=self.M, n_classes=self.n_classes).to(self.device)
        else:
            raise ValueError("prior_type must be 'flow' or 'diffusion'.")
        self.prior_type = prior_type

        self.pca: Optional[PCA] = None
        self.h_mean_: Optional[np.ndarray] = None
        self.h_std_: Optional[np.ndarray] = None
        self.emission: Optional[DenseEmissionParams] = None
        self.train_info: Dict[str, object] = {}

    def infer_h(self, X: np.ndarray, R: np.ndarray) -> Tuple[np.ndarray, np.ndarray]:
        Z = gaussianize_columns(X, R)
        # Missing copula coordinates are mean-imputed at 0 after Gaussianization.
        Z_fit = np.nan_to_num(Z, nan=0.0, posinf=0.0, neginf=0.0)
        max_components = min(self.M, Z_fit.shape[0] - 1, Z_fit.shape[1])
        if max_components < self.M:
            raise ValueError(
                f"M={self.M} exceeds PCA limit {max_components} for training shape {Z_fit.shape}."
            )
        self.pca = PCA(n_components=self.M, svd_solver="full", random_state=GLOBAL_SEED)
        h_raw = self.pca.fit_transform(Z_fit).astype(np.float32)
        self.h_mean_ = h_raw.mean(axis=0, keepdims=True)
        self.h_std_ = h_raw.std(axis=0, keepdims=True) + 1e-6
        h_hat = (h_raw - self.h_mean_) / self.h_std_
        return h_hat.astype(np.float32), Z

    def fit_emissions(
        self,
        Z: np.ndarray,
        R: np.ndarray,
        y: Optional[np.ndarray],
        h_hat: np.ndarray,
    ) -> None:
        n, m = Z.shape
        C = self.n_classes

        W = np.zeros((m, self.M), dtype=np.float32)
        sigma = np.ones(m, dtype=np.float32)
        b = np.zeros((C, m), dtype=np.float32) if C is not None else None

        # Dense loading fit: one OLS problem per feature.
        for j in range(m):
            obs = (R[:, j] == 1) & np.isfinite(Z[:, j])
            if obs.sum() < self.M + (C or 1) + 2:
                continue

            z = Z[obs, j].astype(np.float64)
            H = h_hat[obs].astype(np.float64)

            if C is not None and y is not None:
                yy = y[obs]
                OH = np.zeros((len(yy), C), dtype=np.float64)
                OH[np.arange(len(yy)), yy] = 1.0
                A = np.concatenate([H, OH], axis=1)
                coef, *_ = np.linalg.lstsq(A, z, rcond=None)
                W[j] = coef[: self.M].astype(np.float32)
                b[:, j] = coef[self.M :].astype(np.float32)
                resid = z - A @ coef
            else:
                A = np.concatenate([H, np.ones((H.shape[0], 1))], axis=1)
                coef, *_ = np.linalg.lstsq(A, z, rcond=None)
                W[j] = coef[: self.M].astype(np.float32)
                resid = z - A @ coef

            sigma[j] = float(np.sqrt(max(np.mean(resid ** 2), 1e-4)))

        miss_rate = 1.0 - R.mean(axis=0).astype(np.float32)
        miss_rate_y = None
        if C is not None and y is not None:
            miss_rate_y = np.zeros((C, m), dtype=np.float32)
            for c in range(C):
                mask = y == c
                miss_rate_y[c] = miss_rate if mask.sum() < 2 else 1.0 - R[mask].mean(axis=0)

        self.emission = DenseEmissionParams(
            W=torch.from_numpy(W).to(self.device),
            sigma=torch.from_numpy(sigma).to(self.device),
            b=None if b is None else torch.from_numpy(b).to(self.device),
            miss_rate=torch.from_numpy(np.clip(miss_rate, 1e-4, 1 - 1e-4)).to(self.device),
            miss_rate_y=None if miss_rate_y is None else torch.from_numpy(
                np.clip(miss_rate_y, 1e-4, 1 - 1e-4)
            ).to(self.device),
        )

    def train_prior(
        self,
        h_hat: np.ndarray,
        y: Optional[np.ndarray],
        epochs: int,
        batch_size: int,
        lr: float,
        use_ema: bool,
        ema_decay: float,
    ) -> None:
        h_t = torch.from_numpy(h_hat.astype(np.float32)).to(self.device)
        y_t = None if y is None else torch.from_numpy(y.astype(np.int64)).to(self.device)
        opt = torch.optim.Adam(self.prior.parameters(), lr=lr)
        ema = EMA(self.prior, decay=ema_decay) if self.prior_type == "diffusion" and use_ema else None
        best_loss = float("inf")
        best_state = None
        best_epoch = None
        n = h_t.shape[0]

        for ep in range(1, epochs + 1):
            idx = torch.randint(0, n, (min(batch_size, n),), device=self.device)
            hb = h_t[idx]
            yb = None if y_t is None else y_t[idx]
            if self.prior_type == "flow":
                loss = -self.prior.log_prob(hb, yb).mean()
            else:
                loss = self.prior.training_loss(hb, yb)

            opt.zero_grad()
            loss.backward()
            torch.nn.utils.clip_grad_norm_(self.prior.parameters(), 1.0)
            opt.step()
            if ema is not None:
                ema.update(self.prior)

            val = float(loss.detach().cpu())
            if val < best_loss:
                best_loss = val
                best_epoch = ep
                state = ema.state_dict() if ema is not None else self.prior.state_dict()
                best_state = {k: v.detach().clone() for k, v in state.items()}

        if best_state is not None:
            self.prior.load_state_dict(best_state, strict=True)
        self.train_info = {"best_loss": best_loss, "best_epoch": best_epoch}

    def fit(self, X: np.ndarray, y: Optional[np.ndarray], seed: int) -> "DenseLatentFactorGenerator":
        set_seed(seed)
        X = np.asarray(X, dtype=np.float32)
        R = np.isfinite(X).astype(np.int64)
        self.marginals.fit(X=X, R=R, feature_specs=self.feature_specs, y=y)
        h_hat, Z = self.infer_h(X, R)
        self.fit_emissions(Z=Z, R=R, y=y, h_hat=h_hat)
        self.train_prior(
            h_hat=h_hat,
            y=y,
            epochs=PRIOR_EPOCHS,
            batch_size=PRIOR_BATCH,
            lr=PRIOR_LR,
            use_ema=USE_EMA,
            ema_decay=EMA_DECAY,
        )
        return self

    def _sample_y(self, n: int, y=None) -> Optional[torch.Tensor]:
        if self.n_classes is None:
            return None
        if y is None:
            return torch.randint(0, self.n_classes, (n,), device=self.device)
        if isinstance(y, int):
            return torch.full((n,), y, dtype=torch.long, device=self.device)
        arr = np.asarray(y, dtype=np.int64)
        if arr.shape != (n,):
            raise ValueError(f"Requested labels must have shape ({n},), got {arr.shape}.")
        return torch.from_numpy(arr).to(self.device)

    @torch.no_grad()
    def sample(self, n: int, y=None):
        if self.emission is None:
            raise RuntimeError("Generator has not been fitted.")
        y_t = self._sample_y(n, y)
        h = self.prior.sample(n=n, y=y_t, device=self.device)
        em = self.emission

        mu = h @ em.W.t()
        if self.n_classes is not None and em.b is not None and y_t is not None:
            mu = mu + em.b[y_t]
        z = mu + em.sigma.unsqueeze(0) * torch.randn_like(mu)
        u = erf_normal_cdf(z)

        X = torch.empty((n, self.m), dtype=torch.float32, device=self.device)
        for j in range(self.m):
            if self.use_class_cond_marginals and y_t is not None:
                xj = torch.empty(n, dtype=torch.float32, device=self.device)
                for c in range(self.n_classes):
                    mask = y_t == c
                    if mask.any():
                        xj[mask] = self.marginals.inverse_cdf(u[mask, j], j=j, y=int(c))
            else:
                xj = self.marginals.inverse_cdf(u[:, j], j=j, y=None).float()
            X[:, j] = xj

        if (
            self.n_classes is not None
            and self.use_class_cond_missingness
            and em.miss_rate_y is not None
            and y_t is not None
        ):
            miss_p = em.miss_rate_y[y_t]
        else:
            miss_p = em.miss_rate.unsqueeze(0).expand(n, self.m)
        R = (torch.rand_like(miss_p) > miss_p).long()
        X[R == 0] = float("nan")

        X_out = X.cpu().numpy().astype(np.float32)
        R_out = R.cpu().numpy().astype(np.int64)
        y_out = None if y_t is None else y_t.cpu().numpy().astype(np.int64)
        return X_out, R_out, y_out

    def loading_parameter_count(self) -> int:
        return int(self.m * self.M)


# ============================================================
# Matched generator builders
# ============================================================
def feature_specs_for(X: np.ndarray, names=None):
    if names is None:
        names = [f"f{j}" for j in range(X.shape[1])]
    return [FeatureSpec(name=str(names[j]), kind="continuous") for j in range(X.shape[1])]


def fit_dense(Xtr: np.ndarray, ytr: np.ndarray, seed: int, names=None):
    specs = feature_specs_for(Xtr, names)
    n_classes = int(np.max(ytr)) + 1
    gen = DenseLatentFactorGenerator(
        feature_specs=specs,
        M=M,
        n_classes=n_classes,
        prior_type=PRIOR_TYPE,
        device=DEVICE,
        use_class_cond_marginals=True,
        use_class_cond_missingness=True,
    )
    return gen.fit(Xtr, ytr, seed)


def fit_block(Xtr: np.ndarray, ytr: np.ndarray, seed: int, use_gobs: bool, names=None):
    specs = feature_specs_for(Xtr, names)
    kwargs = dict(
        X=np.asarray(Xtr, dtype=np.float32),
        feature_specs=specs,
        y=np.asarray(ytr, dtype=int),
        M=M,
        blocks=None,
        permute_features=False,
        prior_type=PRIOR_TYPE,
        device=str(DEVICE),
        seed=seed,
        prior_epochs=PRIOR_EPOCHS,
        prior_batch=PRIOR_BATCH,
        prior_lr=PRIOR_LR,
        verbose_every=0,
        save_dir=None,
        save_best=True,
        use_ema=USE_EMA,
        ema_decay=EMA_DECAY,
        return_train_info=False,
        use_gobs=use_gobs,
        gobs_use_cpu_kmeans=False,
    )
    if use_gobs:
        kwargs.update(GOBS_PARAMS)
    return fit_block_subunit_generator(**kwargs)


def fit_method(method: str, Xtr: np.ndarray, ytr: np.ndarray, seed: int, names=None):
    if method == "DenseFactor-PCA":
        return fit_dense(Xtr, ytr, seed, names)
    if method == "BlockSparse-NoGO-BS":
        return fit_block(Xtr, ytr, seed, use_gobs=False, names=names)
    if method == "BSTabDiff-GO-BS":
        return fit_block(Xtr, ytr, seed, use_gobs=True, names=names)
    raise KeyError(method)


# ============================================================
# Evaluation metrics
# ============================================================
def draw_matched_labels(y_train: np.ndarray, n: int, seed: int) -> np.ndarray:
    rng = np.random.RandomState(seed)
    classes, counts = np.unique(y_train, return_counts=True)
    probs = counts / counts.sum()
    return rng.choice(classes, size=n, replace=True, p=probs).astype(int)


def balanced_c2st(
    X_real: np.ndarray,
    X_syn: np.ndarray,
    seed: int,
) -> Dict[str, float]:
    """Balanced classifier two-sample test. AUC-separability is >=0.5."""
    rng = np.random.RandomState(seed)
    nr = X_real.shape[0]
    ns = X_syn.shape[0]
    k = min(nr, ns)
    ir = rng.choice(nr, size=k, replace=False)
    is_ = rng.choice(ns, size=k, replace=False)
    Xc = np.vstack([X_real[ir], X_syn[is_]]).astype(np.float32)
    yc = np.concatenate([np.zeros(k, dtype=int), np.ones(k, dtype=int)])

    sss = StratifiedShuffleSplit(n_splits=1, test_size=0.3, random_state=seed)
    tr, te = next(sss.split(Xc, yc))
    clf = make_lr(seed)
    clf.fit(Xc[tr], yc[tr])
    pred = clf.predict(Xc[te])
    score = positive_class_score(clf, Xc[te])
    raw_auc = safe_auc(yc[te], score)
    sep_auc = max(raw_auc, 1.0 - raw_auc) if np.isfinite(raw_auc) else np.nan
    return {
        "c2st_acc": float(accuracy_score(yc[te], pred)),
        "c2st_auc_raw": raw_auc,
        "c2st_auc_separability": sep_auc,
    }


def ks_w1_1d(x: np.ndarray, y: np.ndarray) -> Tuple[float, float]:
    x = np.sort(np.asarray(x, dtype=float)[np.isfinite(x)])
    y = np.sort(np.asarray(y, dtype=float)[np.isfinite(y)])
    if x.size == 0 or y.size == 0:
        return np.nan, np.nan
    grid = np.sort(np.concatenate([x, y]))
    Fx = np.searchsorted(x, grid, side="right") / x.size
    Fy = np.searchsorted(y, grid, side="right") / y.size
    ks = float(np.max(np.abs(Fx - Fy)))
    if grid.size > 1:
        w1 = float(np.sum(np.abs(Fx[:-1] - Fy[:-1]) * np.diff(grid)))
    else:
        w1 = 0.0
    return ks, w1


def correlation_gap(X_real: np.ndarray, X_syn: np.ndarray) -> Tuple[float, float]:
    Xr = SimpleImputer(strategy="median").fit_transform(X_real).astype(np.float32)
    Xs = SimpleImputer(strategy="median").fit_transform(X_syn).astype(np.float32)
    Cr = np.corrcoef(Xr, rowvar=False)
    Cs = np.corrcoef(Xs, rowvar=False)
    Cr = np.nan_to_num(Cr)
    Cs = np.nan_to_num(Cs)
    iu = np.triu_indices(Cr.shape[0], k=1)
    d = np.abs(Cr[iu] - Cs[iu])
    return float(np.mean(d)), float(np.max(d))


def fidelity_metrics(
    X_real: np.ndarray,
    y_real: np.ndarray,
    X_syn: np.ndarray,
    y_syn: np.ndarray,
    seed: int,
) -> Dict[str, float]:
    ks_vals, w1_vals = [], []
    for j in range(X_real.shape[1]):
        ks, w1 = ks_w1_1d(X_real[:, j], X_syn[:, j])
        ks_vals.append(ks)
        w1_vals.append(w1)

    corr_mean, corr_max = correlation_gap(X_real, X_syn)

    Xr_imp = SimpleImputer(strategy="median").fit_transform(X_real)
    Xs_imp = SimpleImputer(strategy="median").fit_transform(X_syn)
    mi_r = mutual_info_classif(Xr_imp, y_real, random_state=seed)
    mi_s = mutual_info_classif(Xs_imp, y_syn, random_state=seed)
    mi_gap = np.abs(mi_r - mi_s)

    # Synthetic-to-real nearest-neighbor distance.
    sc = StandardScaler().fit(Xr_imp)
    Xr_s = sc.transform(Xr_imp)
    Xs_s = sc.transform(Xs_imp)
    nn = NearestNeighbors(n_neighbors=1, metric="euclidean").fit(Xr_s)
    d, _ = nn.kneighbors(Xs_s)

    c2 = balanced_c2st(X_real, X_syn, seed)

    return {
        "ks_mean": float(np.nanmean(ks_vals)),
        "ks_max": float(np.nanmax(ks_vals)),
        "w1_mean": float(np.nanmean(w1_vals)),
        "w1_max": float(np.nanmax(w1_vals)),
        "pearson_abs_corr_gap_mean": corr_mean,
        "pearson_abs_corr_gap_max": corr_max,
        "mi_gap_mean": float(np.mean(mi_gap)),
        "mi_gap_max": float(np.max(mi_gap)),
        "privacy_nn_mean": float(np.mean(d)),
        "privacy_nn_min": float(np.min(d)),
        **c2,
    }


def count_prior_params(gen) -> int:
    return int(sum(p.numel() for p in gen.prior.parameters()))


def loading_param_count(method: str, gen, m: int) -> int:
    if method == "DenseFactor-PCA":
        return int(m * M)
    return int(m)  # one scalar loading a_j per feature


# ============================================================
# Full-data resource and fidelity evaluation
# ============================================================
def fit_with_resources(method: str, X: np.ndarray, y: np.ndarray, names):
    cleanup()
    set_seed(GLOBAL_SEED)
    proc = psutil.Process(os.getpid()) if psutil is not None else None
    rss_before = proc.memory_info().rss if proc is not None else 0
    if torch.cuda.is_available():
        torch.cuda.reset_peak_memory_stats()
        torch.cuda.synchronize()
    t0 = time.perf_counter()
    gen = fit_method(method, X, y, GLOBAL_SEED, names)
    if torch.cuda.is_available():
        torch.cuda.synchronize()
    elapsed = time.perf_counter() - t0
    rss_after = proc.memory_info().rss if proc is not None else 0
    gpu_peak = torch.cuda.max_memory_allocated() if torch.cuda.is_available() else 0

    y_req = draw_matched_labels(y, N_SYN, GLOBAL_SEED + 999)
    X_syn, R_syn, y_syn = gen.sample(N_SYN, y=y_req)
    X_syn = np.asarray(X_syn, dtype=np.float32)
    y_syn = np.asarray(y_syn, dtype=int)

    fid = fidelity_metrics(X, y, X_syn, y_syn, GLOBAL_SEED)
    row = {
        "method": method,
        "M": M,
        "train_time_sec": elapsed,
        "gpu_peak_GiB": gpu_peak / (1024 ** 3),
        "cpu_rss_GiB": max(rss_before, rss_after) / (1024 ** 3),
        "loading_params": loading_param_count(method, gen, X.shape[1]),
        "prior_params": count_prior_params(gen),
        "total_reported_params": loading_param_count(method, gen, X.shape[1]) + count_prior_params(gen),
        **fid,
    }
    return gen, X_syn, R_syn, y_syn, row


# ============================================================
# Strict 5x5 fold-wise utility evaluation
# ============================================================
def run_cv(X: np.ndarray, y: np.ndarray, names):
    rkf = RepeatedStratifiedKFold(
        n_splits=N_SPLITS,
        n_repeats=N_REPEATS,
        random_state=GLOBAL_SEED,
    )
    rows = []

    for fold, (tr, te) in enumerate(rkf.split(X, y), start=1):
        Xtr, Xte = X[tr], X[te]
        ytr, yte = y[tr], y[te]

        # TRTR is method-independent; compute once per fold.
        trtr = make_lr(GLOBAL_SEED + fold)
        trtr.fit(Xtr, ytr)
        trtr_pred = trtr.predict(Xte)
        trtr_score = positive_class_score(trtr, Xte)
        trtr_acc = float(accuracy_score(yte, trtr_pred))
        trtr_auc = safe_auc(yte, trtr_score)

        y_req = draw_matched_labels(ytr, N_SYN, GLOBAL_SEED + 10000 + fold)

        for mi, method in enumerate(METHODS):
            seed = GLOBAL_SEED + 1000 * (mi + 1) + fold
            cleanup()
            if torch.cuda.is_available():
                torch.cuda.reset_peak_memory_stats()
                torch.cuda.synchronize()
            t0 = time.perf_counter()
            gen = fit_method(method, Xtr, ytr, seed, names)
            if torch.cuda.is_available():
                torch.cuda.synchronize()
            fit_sec = time.perf_counter() - t0
            gpu_peak = torch.cuda.max_memory_allocated() if torch.cuda.is_available() else 0

            Xsyn, _, ysyn = gen.sample(N_SYN, y=y_req)
            Xsyn = np.asarray(Xsyn, dtype=np.float32)
            ysyn = np.asarray(ysyn, dtype=int)

            clf = make_lr(seed)
            clf.fit(Xsyn, ysyn)
            pred = clf.predict(Xte)
            score = positive_class_score(clf, Xte)
            tstr_acc = float(accuracy_score(yte, pred))
            tstr_auc = safe_auc(yte, score)

            c2 = balanced_c2st(Xtr, Xsyn, seed)

            rows.append(
                {
                    "fold": fold,
                    "method": method,
                    "n_train": len(tr),
                    "n_test": len(te),
                    "M": M,
                    "tstr_acc": tstr_acc,
                    "tstr_auc": tstr_auc,
                    "trtr_acc": trtr_acc,
                    "trtr_auc": trtr_auc,
                    "auc_efficiency": tstr_auc / trtr_auc if trtr_auc > 0 else np.nan,
                    "fit_time_sec": fit_sec,
                    "gpu_peak_GiB": gpu_peak / (1024 ** 3),
                    **c2,
                }
            )
            log(
                f"[Fold {fold:02d}/25] {method:22s} | "
                f"TSTR ACC={tstr_acc:.4f} AUC={tstr_auc:.4f} | "
                f"fit={fit_sec:.2f}s"
            )
            del gen, Xsyn, ysyn, clf
            cleanup()

    return pd.DataFrame(rows)


def summarize_cv(fold_df: pd.DataFrame) -> pd.DataFrame:
    metrics = [
        "tstr_acc",
        "tstr_auc",
        "trtr_acc",
        "trtr_auc",
        "auc_efficiency",
        "c2st_acc",
        "c2st_auc_raw",
        "c2st_auc_separability",
        "fit_time_sec",
        "gpu_peak_GiB",
    ]
    rows = []
    for method, g in fold_df.groupby("method", sort=False):
        row = {"method": method, "M": M}
        for metric in metrics:
            row[f"{metric}_mean"] = float(g[metric].mean())
            row[f"{metric}_std"] = float(g[metric].std(ddof=1))
        rows.append(row)
    return pd.DataFrame(rows)


def paired_tests(fold_df: pd.DataFrame) -> pd.DataFrame:
    rows = []
    pivot = fold_df.pivot(index="fold", columns="method", values="tstr_auc")
    for a in METHODS:
        for b in METHODS:
            if METHODS.index(a) >= METHODS.index(b):
                continue
            d = pivot[a] - pivot[b]
            stat = p = np.nan
            if wilcoxon is not None:
                try:
                    stat, p = wilcoxon(d.to_numpy(), zero_method="wilcox", alternative="two-sided")
                except Exception:
                    pass
            rows.append(
                {
                    "method_A": a,
                    "method_B": b,
                    "mean_A_minus_B_tstr_auc": float(d.mean()),
                    "median_A_minus_B_tstr_auc": float(d.median()),
                    "wins_A": int((d > 0).sum()),
                    "ties": int((d == 0).sum()),
                    "wins_B": int((d < 0).sum()),
                    "wilcoxon_stat": stat,
                    "wilcoxon_p": p,
                }
            )
    return pd.DataFrame(rows)


def save_plots(summary: pd.DataFrame, resources: pd.DataFrame):
    order = METHODS
    s = summary.set_index("method").loc[order]

    fig, ax = plt.subplots(figsize=(8.2, 4.6))
    ax.bar(np.arange(len(order)), s["tstr_auc_mean"], yerr=s["tstr_auc_std"], capsize=4)
    ax.set_xticks(np.arange(len(order)))
    ax.set_xticklabels(order, rotation=12, ha="right")
    ax.set_ylabel("TSTR AUC (mean ± std)")
    ax.set_ylim(0.0, 1.05)
    ax.set_title("Colon: dense factor vs block-subunit decoders")
    fig.tight_layout()
    fig.savefig(f"{OUT_PREFIX}_tstr_auc.pdf", bbox_inches="tight")
    fig.savefig(f"{OUT_PREFIX}_tstr_auc.png", dpi=250, bbox_inches="tight")
    plt.close(fig)

    r = resources.set_index("method").loc[order]
    fig, ax = plt.subplots(figsize=(8.2, 4.6))
    ax.bar(np.arange(len(order)), r["train_time_sec"])
    ax.set_xticks(np.arange(len(order)))
    ax.set_xticklabels(order, rotation=12, ha="right")
    ax.set_ylabel("Full-data generator fit time (s)")
    ax.set_title("Computational cost")
    fig.tight_layout()
    fig.savefig(f"{OUT_PREFIX}_runtime.pdf", bbox_inches="tight")
    fig.savefig(f"{OUT_PREFIX}_runtime.png", dpi=250, bbox_inches="tight")
    plt.close(fig)


# ============================================================
# Main
# ============================================================
def main():
    set_seed(GLOBAL_SEED)
    log("=" * 88)
    log("Colon: dense latent factor vs block-subunit ablation")
    log("=" * 88)
    log(f"CUDA_VISIBLE_DEVICES={os.environ.get('CUDA_VISIBLE_DEVICES')}")
    log(f"Torch device={DEVICE}")
    if torch.cuda.is_available():
        log(f"Visible CUDA devices={torch.cuda.device_count()}")
        log(f"Visible device 0={torch.cuda.get_device_name(0)}")
    log(f"Matched configuration: M={M}, prior={PRIOR_TYPE}, epochs={PRIOR_EPOCHS}, n_syn={N_SYN}")

    X, y, names = load_colon_dataset()
    log(f"Loaded Colon: X={X.shape}, y={y.shape}, classes={np.unique(y)}")

    # Full-data fidelity and resource measurements.
    resource_rows = []
    for method in METHODS:
        log(f"\n=== Full-data fit: {method} ===")
        gen, Xsyn, Rsyn, ysyn, row = fit_with_resources(method, X, y, names)
        resource_rows.append(row)
        pd.DataFrame(Xsyn, columns=names).assign(label=ysyn).to_csv(
            f"{OUT_PREFIX}_{method.replace('/', '_').replace(' ', '_')}_synthetic.csv",
            index=False,
        )
        log(pd.Series(row).to_string())
        del gen, Xsyn, Rsyn, ysyn
        cleanup()

    resource_df = pd.DataFrame(resource_rows)
    resource_df.to_csv(f"{OUT_PREFIX}_fidelity_resources.csv", index=False)

    # Strict 5x5 fold-wise TSTR/TRTR/C2ST.
    log("\n=== Strict 5x5 fold-wise Logistic Regression evaluation ===")
    fold_df = run_cv(X, y, names)
    fold_df.to_csv(f"{OUT_PREFIX}_fold_results.csv", index=False)

    summary_df = summarize_cv(fold_df)
    summary_df.to_csv(f"{OUT_PREFIX}_summary.csv", index=False)

    paired_df = paired_tests(fold_df)
    paired_df.to_csv(f"{OUT_PREFIX}_paired_tests.csv", index=False)

    save_plots(summary_df, resource_df)

    log("\n=== CV summary ===")
    log(summary_df.to_string(index=False))
    log("\n=== Full-data fidelity/resources ===")
    log(resource_df.to_string(index=False))
    log("\n=== Paired TSTR-AUC comparisons ===")
    log(paired_df.to_string(index=False))
    log("\nSaved outputs with prefix: " + OUT_PREFIX)


if __name__ == "__main__":
    main()

Loaded BSTabDiff module from: /home/zhabib/bstabdiff_gobs.py
Colon: dense latent factor vs block-subunit ablation
CUDA_VISIBLE_DEVICES=3
Torch device=cuda:0
Visible CUDA devices=1
Visible device 0=NVIDIA RTX A6000
Matched configuration: M=8, prior=flow, epochs=300, n_syn=200
Loaded Colon: X=(62, 2000), y=(62,), classes=[0 1]

=== Full-data fit: DenseFactor-PCA ===
method                       DenseFactor-PCA
M                                          8
train_time_sec                       6.16356
gpu_peak_GiB                        0.029795
cpu_rss_GiB                         1.162235
loading_params                         16000
prior_params                          531680
total_reported_params                 547680
ks_mean                              0.09639
ks_max                               0.28129
w1_mean                             0.148564
w1_max                              0.366948
pearson_abs_corr_gap_mean           0.133779
pearson_abs_corr_gap_max            0.890432
mi_